In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

def parse_benchmark_data(file_path):
    algorithms = {
        "State Vector": {"qubits": [], "time": [], "memory": []},
        "Stabilizer": {"qubits": [], "time": [], "memory": []},
        "Tensor Network": {"qubits": [], "time": [], "memory": []}
    }
    
    current_algorithm = None
    header_seen = False
    
    with open(file_path, 'r') as file:
        for line in file:
            if "=== State Vector Algorithm" in line:
                current_algorithm = "State Vector"
                header_seen = False
                continue
            elif "=== Stabilizer Algorithm" in line:
                current_algorithm = "Stabilizer"
                header_seen = False
                continue
            elif "=== Tensor Network Algorithm" in line:
                current_algorithm = "Tensor Network"
                header_seen = False
                continue
            
            if "Qubits,Time(ms),Memory" in line:
                header_seen = True
                continue
            
            if current_algorithm and header_seen and ',' in line and not line.startswith('#'):
                try:
                    parts = line.strip().split(',')
                    if len(parts) >= 3:
                        qubits = int(parts[0])
                        time_ms = float(parts[1])
                        memory_kb = float(parts[2])
                        
                        algorithms[current_algorithm]["qubits"].append(qubits)
                        algorithms[current_algorithm]["time"].append(time_ms)
                        algorithms[current_algorithm]["memory"].append(memory_kb)
                except ValueError:
                    continue
    
    return algorithms

benchmark_data = parse_benchmark_data('benchmark_results.txt')

dfs = {}
for algo, data in benchmark_data.items():
    if data["qubits"]:
        dfs[algo] = pd.DataFrame({
            'qubits': data["qubits"],
            'time_ms': data["time"],
            'memory_kb': data["memory"]
        })

plt.figure(figsize=(12, 6))
for algo, df in dfs.items():
    plt.plot(df['qubits'], df['time_ms'], marker='o', linewidth=2, label=algo)

plt.xlabel('Number of Qubits')
plt.ylabel('Execution Time (ms)')
plt.title('Execution Time vs. Number of Qubits')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig('execution.png')
plt.show()

plt.figure(figsize=(12, 6))
for algo, df in dfs.items():
    plt.plot(df['qubits'], df['memory_kb'], marker='o', linewidth=2, label=algo)

plt.xlabel('Number of Qubits')
plt.ylabel('Memory Usage (KB)')
plt.title('Memory Usage vs. Number of Qubits')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig('memory.png')
plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for algo, df in dfs.items():
    ax1.semilogy(df['qubits'], df['time_ms'], marker='o', linewidth=2, label=algo)
    ax2.semilogy(df['qubits'], df['memory_kb'], marker='o', linewidth=2, label=algo)

ax1.set_xlabel('Number of Qubits')
ax1.set_ylabel('Execution Time (ms) - Log Scale')
ax1.set_title('Execution Time vs. Number of Qubits')
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.legend()

ax2.set_xlabel('Number of Qubits')
ax2.set_ylabel('Memory Usage (KB) - Log Scale')
ax2.set_title('Memory Usage vs. Number of Qubits')
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()

plt.tight_layout()
plt.show()

if len(dfs) > 1 and "State Vector" in dfs:
    max_common_qubits = min(df['qubits'].max() for df in dfs.values())
    common_qubits = [q for q in range(1, int(max_common_qubits) + 1) 
                    if all(q in df['qubits'].values for df in dfs.values())]
    
    if common_qubits:
        comparison_df = pd.DataFrame({'qubits': common_qubits})
        
        for algo, df in dfs.items():
            comparison_df[f'{algo}_time'] = [df[df['qubits'] == q]['time_ms'].values[0] for q in common_qubits]
            comparison_df[f'{algo}_memory'] = [df[df['qubits'] == q]['memory_kb'].values[0] for q in common_qubits]
        
        for algo in dfs.keys():
            if algo != "State Vector":
                comparison_df[f'{algo}_time_speedup'] = comparison_df['State Vector_time'] / comparison_df[f'{algo}_time']
                comparison_df[f'{algo}_memory_reduction'] = comparison_df['State Vector_memory'] / comparison_df[f'{algo}_memory']
        
        display(comparison_df)
        
        plt.figure(figsize=(12, 6))
        for algo in dfs.keys():
            if algo != "State Vector":
                plt.plot(comparison_df['qubits'], comparison_df[f'{algo}_time_speedup'], 
                         marker='o', linewidth=2, label=f'{algo} Time Speedup')
                plt.plot(comparison_df['qubits'], comparison_df[f'{algo}_memory_reduction'], 
                         marker='s', linestyle='--', linewidth=2, label=f'{algo} Memory Reduction')
        
        plt.xlabel('Number of Qubits')
        plt.ylabel('Efficiency Factor (higher is better)')
        plt.title('Efficiency Comparison (relative to State Vector)')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend()
        plt.tight_layout()
        plt.show()